# The Climate is Changing: How is the Research?

## Exploring Trends in Climate Articles: 2013 to 2023

#### This notebook explores Word2Vec concepts within the corpus.

#### Written by Rafael Alvarado(1) and Caroline Kranefuss(1).

(1) University of Virginia, 2026

In [ ]:
# General imports
import pandas as pd 
import numpy as np 
import os
import sys

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
sns.set_theme(style='white')
pio.renderers.default = 'iframe'

# Project-specific imports
import glob
from glob import glob
import nltk
from nltk import download as nltk_download
nltk_download('vader_lexicon')
nltk_resources = [
    'tokenizers/punkt', 
    'averaged_perceptron_tagger_eng',
    'corpora/stopwords', 
    'help/tagsets'
]

for resource in nltk_resources:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource)
        
# Scikit Learn
from sklearn.manifold import TSNE as tsne

# Gensim
from gensim.corpora import Dictionary
from gensim.models import word2vec

# ----File Stitching----
# If not in repo home folder, cd back 
if os.path.basename(os.getcwd()) != "evolving_sentiment_climate":
    os.chdir('..')
# If a file is in /sources/, access it by telling the system to look at that path as well as current path
sys.path.append(os.path.join(os.getcwd(), 'sources'))
source_dir = "sources"
source_files_paths = glob(f"{source_dir}/*.xml")
# Same for csvs
sys.path.append(os.path.join(os.getcwd(), 'csvs'))
csvs_dir = "csvs"
csvs_files_paths = glob(f"{csvs_dir}/*.csv")
# And for resources
sys.path.append(os.path.join(os.getcwd(), 'resources'))
resources_dir = "resources"
resources_files_paths = glob(f"{resources_dir}/*.csv")


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\student\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\student\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


### Imports

In [7]:
OHCO = ['year', 'mth_day', 'doc_id', 'sent_num', 'token_num']
bags = dict(
    SENTS = OHCO[:4],
    DOCS = OHCO[:3],
    MTH_DAY = OHCO[:2],
    YEAR = OHCO[:1]
)

In [4]:
LIB = pd.read_csv("csvs/LIB/LIB.csv").set_index('doc_id')
TOKENS = pd.read_csv('csvs/CORPUS/CORPUS.csv').set_index(OHCO)

### Convert to Gensim

In [13]:
# Define docs, dictionary
docs = TOKENS.dropna().groupby('doc_id').term_str.apply(list).tolist()
dictionary = Dictionary(docs)

### Generating Word Embeddings

In [50]:
# Setting parameters
w2v_params = dict(
    window = 2,
    vector_size = 200,
    min_count = 50, # THIS LIMITS OUR VOCAB
    workers = 4
)

# Creating model
model = word2vec.Word2Vec(docs, **w2v_params)
model.wv.vectors

# Building df
WV = pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key)
WV.index.name = 'term_str'
WV.to_csv("csvs/W2V/WV.csv")
WV

,0,1,2,3,4,5,6,7,8,9,...,190,191,192,193,194,195,196,197,198,199
term_str,,,,,,,,,,,,,,,,,,,,,
the,0.107879,0.256085,0.056881,0.203700,-0.505221,0.128848,0.277363,0.215088,-0.193355,-0.036565,...,-0.013307,0.054819,-0.137595,0.044506,0.199142,0.274879,0.015971,-0.017654,-0.233217,0.460321
of,0.320275,-0.007395,-0.288183,0.001712,0.198464,0.414614,-0.168640,0.357347,-0.550065,0.408701,...,-0.242051,-0.025166,-0.026825,0.004375,0.797239,-0.068289,-0.043750,-0.124182,-0.357274,-0.203070
and,-0.184996,0.102881,-0.083532,0.102776,0.161272,0.616071,-0.098258,0.139350,-0.183768,0.171155,...,0.020871,0.053866,0.178012,0.082246,0.227645,0.088381,-0.270416,-0.084524,0.055285,0.154860
in,0.125248,0.138807,0.171537,0.049841,0.405527,0.373281,0.032566,0.067545,-0.496027,0.472899,...,-0.156431,0.130046,-0.432751,0.116781,0.472455,-0.142533,-0.384445,-0.051532,0.113414,-0.079223
to,-0.322428,-0.120037,0.140650,0.234071,0.408531,0.333874,0.013052,0.327789,-0.185881,0.186105,...,-0.133510,0.215951,0.753347,0.508020,0.393611,-0.126511,-0.290605,0.461942,0.560121,0.200398
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
limitations,0.128001,-0.035168,0.044661,0.023763,0.178965,-0.057743,-0.020652,0.220412,-0.053013,0.035616,...,0.073421,-0.039576,-0.124972,-0.102403,-0.016877,0.034917,0.049860,0.004091,-0.138874,0.016365
composed,0.042045,-0.050764,0.021131,0.062602,0.155015,0.006668,0.020990,0.228534,-0.147621,-0.023963,...,0.060501,-0.004251,-0.034064,-0.031309,0.032900,0.071483,0.071023,-0.012965,-0.123930,0.027291
curve,-0.002419,-0.090084,-0.023491,0.027467,0.176095,-0.013650,0.007597,0.224248,-0.111493,0.052457,...,0.095629,0.063946,-0.025126,-0.034956,0.080194,0.077337,0.051196,-0.007405,-0.129743,0.007188


### Visualization: tSNE

In [ ]:
# Set perplexity to 50
PP = 50

# Engine settings
tsne_engine = tsne(
    perplexity=PP, 
    n_components=2, 
    init='pca', 
    max_iter=2500, 
    random_state=23
)

# Data frame
TSNE = pd.DataFrame(
    tsne_engine.fit_transform(WV), 
    columns=['x','y'], 
    index=WV.index)

TSNE

,x,y
term_str,,
the,-6.926507,2.312721
of,-13.830667,-16.575411
and,-15.272699,-10.982263
in,-15.797554,-18.889769
to,37.624279,6.062204
...,...,...
limitations,13.994779,7.450169
composed,-0.243150,3.974092
curve,-10.743078,3.099014


In [52]:
# Plotting TSNE
px.scatter(TSNE.reset_index(), 'x', 'y', 
        text='term_str', 
        hover_name='term_str',  
        # size='n',
        height=1000,
        width=1200)\
    .update_traces(
        mode='markers+text', 
        textfont=dict(color='black', size=14, family='Arial'),
        textposition='top center')

In [12]:
def complete_analogy(A, B, C, n=5):
    try:
        cols = ['term', 'sim']
        return pd.DataFrame(model.wv.most_similar(positive=[B, C], negative=[A])[0:n], columns=cols)
    except KeyError as e:
        print('Error:', e)
        return None
    
def get_most_similar(positive, negative=None):
    return pd.DataFrame(model.wv.most_similar(positive, negative), columns=['term', 'sim'])

In [45]:
complete_analogy('land','soil','ocean')

,term,sim
0,concentrations,0.876507
1,doc,0.865554
2,ambient,0.847172
3,fluxes,0.826772
4,elevated,0.824137


In [31]:
complete_analogy('insect','bee','animal')

,term,sim
0,new,0.912727
1,particular,0.908570
2,endemic,0.903878
3,additional,0.898990
4,unknown,0.892832


In [ ]:
complete_analogy('coral','reef','sparrow')

,term,sim
0,trend,0.913089
1,determining,0.909818
2,record,0.909192
3,challenge,0.898791
4,attack,0.897182


In [48]:
complete_analogy('species','population','blockchain')

,term,sim
0,city,0.766628
1,global,0.692486
2,ecosystem,0.684174
3,development,0.683593
4,state,0.676762


Analysis:

For these analogies, I chose to explore some concepts from the principal components. The analogies were not very accuracte: 

Land is to soil as ocean is to ___ --> I was expecting to see water, since land is made up of soil and the ocean is made up of water, but the only mildly close word I found was ambient, which could refer to water temperature, but that is a bit of a stretch.

Insect is to be as animal is to __ --> I hypothesized I'd see fox or another animal, since a bee is a type of insect, but got mostly random words.

Coral is to reef as sparrow is to __ --> My guess was that I'd see tree or another structure in which sparrows live (even a figurative structure like a flok), given that coral grows on/in reefs, but again found random words.

Species is to population as blockchain is to __ --> This one was less well laid out but I wanted to explore the bio vs technical axis that I've hypothesized about throughout this project. This did yield some potentially interesting results, with words like city and global relating to blockchain (a new technology) in the same way that species and populations are similar words.